# COP · Análisis de errores y explicabilidad operativa

El objetivo es entender dónde se equivoca el modelo en validation, sin convertir una explicación en causalidad ni abrir el test.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

study_dir = Path(
    subprocess.check_output(
        [sys.executable, "scripts/run_academic_study.py"], text=True
    ).strip()
)
predictions = pd.read_csv(study_dir / "validation-predictions.csv")
results = json.loads((study_dir / "study-results.json").read_text())
predictions.head()

In [ ]:
model = "lightgbm"
probability = predictions[f"{model}_validation_probability"]
predictions["uncertainty"] = (probability - 0.5).abs()
predictions["prediction"] = (probability >= 0.5).astype(int)
predictions["error_type"] = np.select(
    [
        (predictions.target == 1) & (predictions.prediction == 0),
        (predictions.target == 0) & (predictions.prediction == 1),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)
predictions.sort_values("uncertainty").head(20)[
    ["ID", "target", "prediction", "error_type", "uncertainty"]
]

In [ ]:
fig = px.histogram(
    predictions,
    x=f"{model}_validation_probability",
    color="target",
    nbins=25,
    barmode="overlay",
    opacity=0.65,
    title="Distribución de probabilidades por target",
)
fig.update_layout(xaxis_title="Probabilidad estimada", yaxis_title="Casos")
fig.show()

## Límites

Este análisis describe comportamiento en validation. No prueba causalidad, fairness, estabilidad temporal ni utilidad para decisiones reales.